## Задача 1 (БЧХ-код (15,7), d >= 5)

Для БЧХ-кода $(15,7)$ с минимальным расстоянием 5:
- построить порождающую матрицу в двоичном виде;
- показать процедуру декодирования для принятой из канала последовательности `111001100011000`.



In [5]:
PRIM_POLY = 0b10011  # p(x)=x^4+x+1, как в конспектах 8-10


def gf_add(a, b):
    return a ^ b  # сложение в GF(16)


def gf_mul(a, b):  # умножение в GF(16)
    res = 0
    aa = a
    bb = b
    while bb:
        if bb & 1:
            res ^= aa
        bb >>= 1
        aa <<= 1
        if aa & 0b10000:
            aa ^= PRIM_POLY
    return res & 0b1111


def build_tables():  # строим таблицы степеней примитивного элемента alpha и таблицу логарифмов в GF(16)
    alpha = (
        0b0010  # примитивный элемент поля GF(16) — это x, его двоичное представление
    )
    alog = [1]  # alog[i] = alpha^i, начинаем с alpha^0 = 1
    for _ in range(1, 15):
        # последовательно перемножаем на alpha, чтобы получить степени alpha от 1 до 14
        alog.append(gf_mul(alog[-1], alpha))

    log = {1: 0}  # log[v] = i, то есть v=alpha^i, обратная таблица логарифмов
    for i, v in enumerate(alog):
        # для каждого элемента из таблицы степеней записываем степень
        log[v] = i
    return alog, log  # возвращаем таблицу степеней и таблицу логарифмов


ALOG, LOG = build_tables()


def gf_inv(a):
    if a == 0:
        # Если a=0, обратного элемента не существует
        raise ZeroDivisionError("Обратный элемент для 0 не существует")
    # Вычисляем обратный элемент в GF(16):
    # если a=alpha^i, то обратный alpha^{15-i} (в порядке циклической группы)
    # (15 - LOG[a]) % 15 — это показатель степени для обратного элемента
    return ALOG[(15 - LOG[a]) % 15]


def gf_div(a, b):
    return gf_mul(a, gf_inv(b))


def poly_mul_bin(p, q):
    """Перемножение бинарных полиномов (коэф. 0/1, low->high)."""
    res = [0] * (
        len(p) + len(q) - 1
    )  # Результат: массив длиной сумма степеней минус 1, с нулями
    for i, pi in enumerate(p):  # Для каждого коэффициента pi многочлена p
        if pi:  # Если коэффициент равен 1 (иначе можно пропустить)
            for j, qj in enumerate(q):  # Для каждого коэффициента qj многочлена q
                if qj:  # Если коэффициент равен 1
                    res[i + j] ^= 1  # Складываем по mod2 в соответствующей позиции
    return res  # Возвращаем получившийся массив-результат


def poly_div_rem_bin(dividend, divisor):
    """Остаток от деления binary poly на binary poly."""
    tmp = dividend[:]  # Копируем делимое, чтобы не портить исходный массив
    deg_d = len(divisor) - 1  # Степень делителя
    # Начинаем с самого старшего бита делимого и двигаемся к младшему,
    # пока можем "вычитать" делитель
    for i in range(len(tmp) - 1, deg_d - 1, -1):
        if tmp[i]:  # Если текущий разряд равен 1, делитель надо вычитать (по mod2)
            for j, dj in enumerate(divisor):  # Проходим по коэффициентам делителя
                if dj:  # Если коэффициент делителя равен 1
                    tmp[
                        i - deg_d + j
                    ] ^= 1  # Выполняем xor для соответствующего разряда
    return tmp[:deg_d]  # Оставляем только младшие разряды — это остаток


def bch_15_7_generator():  # функция для генерации порождающего полинома
    m1 = [1, 1, 0, 0, 1]  # Минимальный полином для класса 1: 1 + x + x^4
    m3 = [1, 1, 1, 1, 1]  # Минимальный полином для класса 3: 1 + x + x^2 + x^3 + x^4
    g = poly_mul_bin(
        m1, m3
    )  # Перемножаем минимальные полиномы, получаем g(x) степени 8
    return m1, m3, g  # Возвращаем оба минимальных полинома и порождающий полином кода


def generator_matrix_cyclic(g, n=15, k=7):
    rows = []  # Здесь будем накапливать строки генераторной матрицы
    for i in range(k):  # Для каждой строки (всего k строк)
        # Формируем строку: сначала i нулей, затем порождающий полином g, затем добиваем нулями справа до длины n
        row = [0] * i + g + [0] * (n - len(g) - i)
        rows.append(row)  # Добавляем строку в список строк
    return rows  # Возвращаем получившуюся генераторную матрицу (список строк)


def print_matrix(rows, title):
    print(title)
    for r in rows:
        print(" ", "".join(map(str, r)))


def syndromes_for_word(r_bits):
    s1 = 0  # Инициализируем первый синдром S1
    s3 = 0  # Инициализируем второй синдром S3
    for i, bit in enumerate(
        r_bits
    ):  # Перебираем все разряды принятого слова (коэффициенты)
        if bit:  # Если бит равен 1 (т.е. ненулевой коэффициент)
            s1 ^= ALOG[i % 15]  # Добавляем к синдрому S1 alpha^{i} (по mod 2)
            s3 ^= ALOG[(3 * i) % 15]  # Добавляем к синдрому S3 alpha^{3*i} (по mod 2)
    return s1, s3  # Возвращаем оба синдрома


def decode_bch_2errors(received_str):
    # Преобразуем принятую строку в список бит (порядок: правый бит -> c0)
    r = [int(ch) for ch in received_str[::-1]]
    # Считаем синдромы S1 и S3 для принятого слова (определяют наличие и расположение ошибок)
    s1, s3 = syndromes_for_word(r)

    print("\nДекодирование принятой последовательности:")
    print(f"  r = {received_str}")
    print("  (соглашение: правый бит соответствует коэффициенту при x^0)")
    print(f"  S1 = {s1:04b}" + (f" = alpha^{LOG[s1]}" if s1 else " = 0"))
    print(f"  S3 = {s3:04b}" + (f" = alpha^{LOG[s3]}" if s3 else " = 0"))

    # Если оба синдрома равны нулю — ошибок нет, возвращаем исходное слово
    if s1 == 0 and s3 == 0:
        print("  Синдромы нулевые -> ошибок нет.")
        return received_str, []

    # По конспекту:
    # x+y=S1, x^3+y^3=S3, корни уравнения z^2 + S1*z + (S3/S1 + S1^2) = 0 дают позиции ошибок
    # Вычисляем слагаемое c: c = S3/S1 + S1^2 по сложению/умножению/делению в поле
    c_term = gf_add(gf_div(s3, s1), gf_mul(s1, s1))
    print(
        f"  c = S3/S1 + S1^2 = {c_term:04b}"
        + (f" = alpha^{LOG[c_term]}" if c_term else " = 0")
    )
    print("  Решаем уравнение: z^2 + S1*z + c = 0")

    roots = []  # Здесь будем искать корни уравнения (кандидаты на ошибки)
    for z in range(16):  # Перебираем все элементы поля (0...15, закодированные int'ом)
        # Проверяем, обращается ли выражение в ноль при данном z
        val = gf_add(gf_add(gf_mul(z, z), gf_mul(s1, z)), c_term)
        if val == 0:
            roots.append(z)

    # Корни не должны быть нулём — иначе это не позиции ошибок
    roots = [z for z in roots if z != 0]
    # Для каждого корня находим дискретный логарифм по базе alpha — это и есть позиция ошибки
    error_pos = [LOG[z] for z in roots]
    error_pos = sorted(error_pos)
    print(
        "  Корни z:",
        [f"{z:04b}" + (f"(alpha^{LOG[z]})" if z in LOG else "") for z in roots],
    )
    print("  Позиции ошибок (нумерация от 0, справа налево):", error_pos)

    # Исправляем найденные ошибки: инвертируем соответствующие биты в принятом слове
    corrected = r[:]
    for p in error_pos:
        corrected[p] ^= 1

    # Собираем исправленное слово обратно в строку (старший бит — слева)
    decoded = "".join(map(str, corrected[::-1]))
    print(f"  Исправленное слово: {decoded}")
    return decoded, error_pos


def main():
    # Генерируем порождающие многочлены m1(x), m3(x) и порождающий многочлен g(x) для BCH(15,7)
    m1, m3, g = bch_15_7_generator()

    # Выводим информацию о построении кода BCH(15,7) с минимальным расстоянием d>=5
    print("Построение BCH(15,7), d>=5")
    print("  m1(x) = 1 + x + x^4")
    print("  m3(x) = 1 + x + x^2 + x^3 + x^4")
    # Выводим порождающий многочлен g(x) в явном виде
    print(
        "  g(x) = m1(x)*m3(x) =",
        " + ".join(f"x^{i}" if i else "1" for i, c in enumerate(g) if c),
    )
    # Выводим вектор коэффициентов порождающего многочлена g(x)
    print("  Вектор коэффициентов g(x) (от x^0 к x^8):", g)

    # Получаем порождающую матрицу G в циклической форме для параметров (n=15, k=7)
    G = generator_matrix_cyclic(g, n=15, k=7)
    # Выводим порождающую матрицу
    print_matrix(G, "\nПорождающая матрица G (циклическая форма, двоичный вид):")

    # Пример принятого кодового слова с потенциальными ошибками (строка длиной 15 бит)
    received = "111001100011000"
    # Декодируем последовательность, исправляем не более двух ошибок
    decoded, err_pos = decode_bch_2errors(received)

    # Проверяем, что исправленное слово действительно делится на g(x) без остатка
    decoded_bits = [
        int(ch) for ch in decoded[::-1]
    ]  # Преобразуем строку в список битов (от младшего к старшему)
    rem = poly_div_rem_bin(decoded_bits, g)  # Получаем остаток от деления на g(x)
    print("\nПроверка делимости исправленного слова на g(x):")
    print(
        "  Остаток:",
        rem,
        "->",
        "кодовое слово" if all(v == 0 for v in rem) else "НЕ кодовое слово",
    )
    # Выводим итоговую информацию: найденные ошибки и итоговое декодированное слово
    print("\nИтог:")
    print("  Найдены ошибки в позициях:", err_pos)
    print("  Декодированное слово:", decoded)


main()

Построение BCH(15,7), d>=5
  m1(x) = 1 + x + x^4
  m3(x) = 1 + x + x^2 + x^3 + x^4
  g(x) = m1(x)*m3(x) = 1 + x^4 + x^6 + x^7 + x^8
  Вектор коэффициентов g(x) (от x^0 к x^8): [1, 0, 0, 0, 1, 0, 1, 1, 1]

Порождающая матрица G (циклическая форма, двоичный вид):
  100010111000000
  010001011100000
  001000101110000
  000100010111000
  000010001011100
  000001000101110
  000000100010111

Декодирование принятой последовательности:
  r = 111001100011000
  (соглашение: правый бит соответствует коэффициенту при x^0)
  S1 = 1111 = alpha^12
  S3 = 1001 = alpha^14
  c = S3/S1 + S1^2 = 1110 = alpha^11
  Решаем уравнение: z^2 + S1*z + c = 0
  Корни z: ['0001(alpha^0)', '1110(alpha^11)']
  Позиции ошибок (нумерация от 0, справа налево): [0, 11]
  Исправленное слово: 111101100011001

Проверка делимости исправленного слова на g(x):
  Остаток: [0, 0, 0, 0, 0, 0, 0, 0] -> кодовое слово

Итог:
  Найдены ошибки в позициях: [0, 11]
  Декодированное слово: 111101100011001
